# Realistic Level 0 single-seed diagnostics

This notebook analyzes the isolated GPT-2-BPE AdamW baseline. Test metrics are shown only for the final and validation-selected checkpoints; the training curve itself uses fixed train and validation probes.


In [ ]:
import json
import os
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path(os.getenv("NANOGPT_LEVEL0_RESULTS_ROOT", "/tmp/nanogpt-level0-bpe/results"))
OPTIMIZER = os.getenv("NANOGPT_LEVEL0_NOTEBOOK_OPTIMIZER", "adamw")
SEED = int(os.getenv("NANOGPT_LEVEL0_NOTEBOOK_SEED", "1337"))
RUN = ROOT / f"{OPTIMIZER}_seed_{SEED}"
REPORT = RUN / "plots"
REPORT.mkdir(parents=True, exist_ok=True)

metrics = pd.read_csv(RUN / "metrics.csv")
manifest = json.loads((RUN / "manifest.json").read_text())
complete = json.loads((RUN / "run_complete.json").read_text())
final = json.loads((RUN / "final_metrics.json").read_text())
selected = json.loads((RUN / "selected_checkpoint_metrics.json").read_text())

print(f"Run: {RUN}")
print(f"Parameters: {manifest['parameter_count']:,}")
print(f"Model: {manifest['model_config']}")
print(f"Final step: {complete['final_step']:,}")
print(f"Final validation loss: {complete['final_validation_loss']:.4f}")
print(f"Final test loss: {final['test_loss']:.4f}")
print(f"Selected step: {selected['selected_step']:,}")
print(f"Selected test loss: {selected['test_loss']:.4f}")
metrics.head()


In [ ]:
def save_current(name):
    plt.tight_layout()
    plt.savefig(REPORT / name, dpi=180, bbox_inches="tight")
    plt.show()

plt.figure(figsize=(10, 6))
plt.plot(metrics.step, metrics.train_loss, label="train probe")
plt.plot(metrics.step, metrics.val_loss, label="validation probe")
plt.axvline(selected["selected_step"], linestyle="--", label="selected checkpoint")
plt.xlabel("optimizer step")
plt.ylabel("cross-entropy loss")
plt.title(f"{OPTIMIZER} seed {SEED}: loss")
plt.grid(alpha=0.25)
plt.legend()
save_current("loss.png")


In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(metrics.step, metrics.train_perplexity, label="train probe")
plt.plot(metrics.step, metrics.val_perplexity, label="validation probe")
plt.scatter([selected["selected_step"]], [selected["test_perplexity"]], marker="x", s=80, label="selected test")
plt.scatter([final["step"]], [final["test_perplexity"]], marker="x", s=80, label="final test")
plt.xlabel("optimizer step")
plt.ylabel("perplexity")
plt.title(f"{OPTIMIZER} seed {SEED}: perplexity")
plt.grid(alpha=0.25)
plt.legend()
save_current("perplexity.png")


In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(metrics.step, 100 * metrics.train_accuracy, label="train top-1 token accuracy")
plt.plot(metrics.step, 100 * metrics.val_accuracy, label="validation top-1 token accuracy")
plt.scatter([selected["selected_step"]], [100 * selected["test_accuracy"]], marker="x", s=80, label="selected test")
plt.scatter([final["step"]], [100 * final["test_accuracy"]], marker="x", s=80, label="final test")
plt.xlabel("optimizer step")
plt.ylabel("exact next-BPE-token accuracy (%)")
plt.title(f"{OPTIMIZER} seed {SEED}: token accuracy")
plt.grid(alpha=0.25)
plt.legend()
save_current("token_accuracy.png")


In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(metrics.step, metrics.train_bits_per_token, label="train")
plt.plot(metrics.step, metrics.val_bits_per_token, label="validation")
plt.scatter([selected["selected_step"]], [selected["test_bits_per_token"]], marker="x", s=80, label="selected test")
plt.scatter([final["step"]], [final["test_bits_per_token"]], marker="x", s=80, label="final test")
plt.xlabel("optimizer step")
plt.ylabel("bits per BPE token")
plt.title(f"{OPTIMIZER} seed {SEED}: bits per token")
plt.grid(alpha=0.25)
plt.legend()
save_current("bits_per_token.png")


In [ ]:
figure, axes = plt.subplots(2, 2, figsize=(12, 9))
axes[0, 0].plot(metrics.step, metrics.learning_rate)
axes[0, 0].set_title("warmup + cosine learning rate")
axes[0, 1].plot(metrics.step, metrics.grad_norm)
axes[0, 1].set_title("gradient norm")
axes[1, 0].plot(metrics.step, metrics.weight_norm)
axes[1, 0].set_title("weight norm")
axes[1, 1].plot(metrics.step, metrics.val_generalization_gap)
axes[1, 1].axhline(0, linewidth=1)
axes[1, 1].set_title("validation loss − train loss")
for axis in axes.ravel():
    axis.set_xlabel("optimizer step")
    axis.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(REPORT / "optimization_diagnostics.png", dpi=180, bbox_inches="tight")
plt.show()


In [ ]:
weightwatcher_files = sorted(RUN.glob("weightwatcher_step_*.csv"))
print(f"WeightWatcher checkpoints: {len(weightwatcher_files)}")
if weightwatcher_files:
    ww = pd.concat([pd.read_csv(path) for path in weightwatcher_files], ignore_index=True)
    ww["alpha"] = pd.to_numeric(ww.get("alpha"), errors="coerce")
    if "matrix_name" not in ww:
        source = "longname" if "longname" in ww else "name"
        ww["matrix_name"] = ww[source].astype(str)
    ww["matrix_type"] = ww.matrix_name.str.replace(r"^L\d+_", "", regex=True)
    ww["block"] = ww.matrix_name.str.extract(r"^(L\d+)", expand=False)
    ww = ww[np.isfinite(ww.alpha)].copy()
    display_columns = [column for column in ["step", "matrix_name", "alpha", "D", "xmin", "num_evals"] if column in ww]
    display(ww[display_columns].head())
else:
    ww = pd.DataFrame()


In [ ]:
if not ww.empty:
    plt.figure(figsize=(10, 6))
    aggregate = ww.groupby("step").alpha.agg(["median", "min", "max"]).reset_index()
    plt.plot(aggregate.step, aggregate["median"], label="median layer alpha")
    plt.fill_between(aggregate.step, aggregate["min"], aggregate["max"], alpha=0.15, label="layer range")
    plt.axhline(2.0, linestyle="--", label="alpha = 2")
    plt.xlabel("optimizer step")
    plt.ylabel("WeightWatcher alpha")
    plt.title(f"{OPTIMIZER} seed {SEED}: aggregate alpha trajectory")
    plt.grid(alpha=0.25)
    plt.legend()
    save_current("alpha_aggregate.png")


In [ ]:
if not ww.empty:
    for matrix_type, group in ww.groupby("matrix_type"):
        plt.figure(figsize=(10, 6))
        for matrix_name, trajectory in group.groupby("matrix_name"):
            trajectory = trajectory.sort_values("step")
            plt.plot(trajectory.step, trajectory.alpha, marker="o", markersize=3, label=matrix_name)
        plt.axhline(2.0, linestyle="--", linewidth=1, label="alpha = 2")
        plt.xlabel("optimizer step")
        plt.ylabel("WeightWatcher alpha")
        plt.title(f"{OPTIMIZER} seed {SEED}: {matrix_type}")
        plt.grid(alpha=0.25)
        plt.legend(ncol=2, fontsize=8)
        save_current(f"alpha_{matrix_type.lower()}.png")

print(f"Saved plots to {REPORT}")
